# Verification artifact export

Run the cell from the repository root, or set `RUN` to the completed output directory. It collects existing files without fitting models or modifying the recorded results.

In [1]:
from pathlib import Path
import csv
import hashlib
import io
import json
import zipfile

RUN = Path(globals().get('RUN_DIR', Path.cwd() / 'Outputs' / 'CHBR_R1_v3')).resolve()
SCOPE = 'core'
DESTINATION = RUN / 'results_verification.zip'


def digest_file(path):
    value = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def csv_bytes(rows, fields):
    stream = io.StringIO(newline='')
    writer = csv.DictWriter(stream, fieldnames=fields, lineterminator='\n')
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue().encode('utf-8')


if SCOPE not in {'core', 'complete'}:
    raise ValueError('SCOPE must be core or complete.')
if not (RUN / 'tables' / 'run_summary.json').is_file():
    raise FileNotFoundError(f'No completed run at {RUN}. Set RUN to the existing output directory.')
summary = json.loads((RUN / 'tables' / 'run_summary.json').read_text(encoding='utf-8'))
config = json.loads((RUN / 'tables' / 'config.json').read_text(encoding='utf-8'))
if summary.get('status') != 'complete':
    raise ValueError('The run summary does not report completion.')

required = {
    'manifest.csv', 'tables/config.json', 'tables/run_summary.json',
    'tables/cyber_row_audit.csv.gz', 'tables/cyber_split_membership.csv.gz',
    'tables/video_split_membership.csv.gz', 'tables/development_holdout_membership.csv.gz',
    'tables/graph_nodes.csv.gz', 'tables/graph_components.csv',
    'tables/fusion_window_table.csv', 'tables/fusion_window_validation_table.csv',
    'tables/fusion_contribution_audit.csv.gz',
    'tables/predictions/fusion_predictions.csv.gz',
    'tables/predictions/pairing_0_test.csv.gz',
    'tables/predictions/pairing_0_validation.csv.gz',
    'tables/predictions/pairing_0_membership.csv.gz',
    'tables/predictions/paired_0_predictions.csv.gz',
}
if config.get('graph_store_edges', False):
    required.add('tables/graph_edges.npz')
for branch, names, feature_sets in (
    ('cyber', config['models_to_run'], ['full']),
    ('video', config['video_models'], ['full', 'ablated']),
):
    for name in names:
        for feature_set in feature_sets:
            for partition in ('development', 'test'):
                required.add(f'tables/predictions/{branch}_{feature_set}_{name}_{partition}.csv.gz')
    for partition in ('threshold_selection', 'test'):
        required.add(f'tables/predictions/{branch}_events_{partition}.csv.gz')
missing = sorted(name for name in required if not (RUN / name).is_file())
if missing:
    raise FileNotFoundError('Required files are absent from this output directory:\n' + '\n'.join(missing))

source_manifest = {}
with (RUN / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    for row in csv.DictReader(stream):
        source_manifest[row['file'].replace('\\', '/')] = row

stress_examples = {
    'tables/predictions/sensitivity/offset_0.csv.gz',
    'tables/predictions/sensitivity/missing_both_0.5.csv.gz',
    'tables/predictions/sensitivity/missing_both_0.5_observation_mask.csv.gz',
    'tables/predictions/sensitivity/window_120.csv.gz',
    'tables/predictions/sensitivity/ordering_random_0.csv.gz',
    'tables/predictions/sensitivity/ordering_random_0_cyber_event_mapping.csv.gz',
}


def include(relative):
    name = relative.as_posix()
    if relative.suffix in {'.zip', '.tmp'} or 'models' in relative.parts:
        return False
    if name in {'manifest.csv', 'tables/figure_index.csv'}:
        return False
    if SCOPE == 'complete':
        return True
    if name in required or name in stress_examples:
        return True
    if 'predictions' in relative.parts:
        return relative.parent.as_posix() == 'tables/predictions' and relative.name.startswith('diagnostic_latent_')
    if 'graph_sensitivity' in relative.parts and relative.suffix == '.gz':
        return False
    return relative.suffix not in {'.gz', '.npz', '.joblib'}


paths = [path for path in sorted(RUN.rglob('*')) if path.is_file() and include(path.relative_to(RUN))]
if not paths:
    raise RuntimeError('No files selected for export.')
figure_rows = [{'file': path.relative_to(RUN).as_posix()} for path in sorted((RUN / 'figures').glob('*.png'))]
virtual = {
    'tables/figure_index.csv': csv_bytes(figure_rows, ['file']),
    'audit/source_manifest.csv': (RUN / 'manifest.csv').read_bytes(),
    'audit/export_scope.json': json.dumps({
        'scope': SCOPE,
        'source_directory': str(RUN),
        'source_files_modified': False,
        'models_included': False,
        'core_scope': 'Primary branch predictions, first fixed pairing, split memberships, primary graph, selected stress records, and aggregate outputs.',
        'complete_scope': 'All non-model research artifacts; archives excluded.',
        'figure_index': 'Reconstructed from existing PNG files; portable relative paths.',
    }, indent=2).encode('utf-8'),
}
manifest_rows = []
for path in paths:
    name = path.relative_to(RUN).as_posix()
    value = digest_file(path)
    previous = source_manifest.get(name)
    if previous and previous.get('sha256') and value != previous['sha256']:
        raise ValueError(f'File differs from the recorded run manifest: {name}. Preserve the original run before export.')
    manifest_rows.append({'file': name, 'bytes': path.stat().st_size, 'sha256': value})
for name, data in virtual.items():
    manifest_rows.append({'file': name, 'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()})
virtual['manifest.csv'] = csv_bytes(sorted(manifest_rows, key=lambda row: row['file']), ['file', 'bytes', 'sha256'])

temporary = DESTINATION.with_suffix('.zip.tmp')
try:
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in paths:
            compression = zipfile.ZIP_STORED if path.suffix in {'.gz', '.npz'} else zipfile.ZIP_DEFLATED
            archive.write(path, path.relative_to(RUN).as_posix(), compress_type=compression)
        for name, data in virtual.items():
            archive.writestr(name, data)
    with zipfile.ZipFile(temporary) as archive:
        corrupt = archive.testzip()
        if corrupt is not None:
            raise IOError(f'Archive integrity check failed: {corrupt}')
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError('Duplicate archive entry.')
    temporary.replace(DESTINATION)
except BaseException:
    if temporary.exists():
        temporary.unlink()
    raise

print(f'Archive: {DESTINATION}')
print(f'Scope: {SCOPE}; files: {len(paths) + len(virtual)}; PNG figures indexed: {len(figure_rows)}')
print(f'Size: {DESTINATION.stat().st_size / 1024**2:.1f} MiB')
print(f'SHA-256: {digest_file(DESTINATION)}')
print('Existing research outputs were not modified. No models were fitted and no experiments were repeated.')


Archive: F:\MBZUAI UAE\7-Elsevier-CHBR-Human-centered\GitHub-v2\Outputs\CHBR_R1_v3\results_verification.zip
Scope: core; files: 307; PNG figures indexed: 18
Size: 51.2 MiB
SHA-256: 6f6edcdc6d7767f793d08d7db323c39e9226b0f20c1f035063b058c545e06e5d
Existing research outputs were not modified. No models were fitted and no experiments were repeated.
